In [3]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    f1_score, classification_report, balanced_accuracy_score,
    confusion_matrix, matthews_corrcoef, cohen_kappa_score, accuracy_score, precision_score, recall_score
)
from imblearn.over_sampling import SMOTE

In [4]:
with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/processed_data/wesad/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

print(data.keys())

df           = data['df']
X            = data['X']
y            = data['y']
groups       = data['groups']
feature_cols = data['feature_cols']

print(df.shape)
print(y.value_counts())

dict_keys(['df', 'X', 'y', 'groups', 'feature_cols'])
(27543, 139)
label
0    17577
1     9966
Name: count, dtype: int64


In [5]:
print([col for col in feature_cols if 'acc' in col.lower()])
print([col for col in feature_cols if 'hr' in col.lower()])
print([col for col in feature_cols if 'ibi' in col.lower() or 'rmssd' in col.lower() or 'sdnn' in col.lower()])

['acc_wrist_mean', 'acc_wrist_std', 'acc_wrist_min', 'acc_wrist_max', 'acc_wrist_p25', 'acc_wrist_p75', 'acc_wrist_skew', 'acc_wrist_kurtosis', 'acc_chest_mean', 'acc_chest_std', 'acc_chest_min', 'acc_chest_max', 'acc_chest_p25', 'acc_chest_p75', 'acc_chest_skew', 'acc_chest_kurtosis']
['hr_mean', 'hr_std', 'hr_min', 'hr_max', 'hr_p25', 'hr_p75', 'hr_skew', 'hr_kurtosis', 'ecg_hr_mean', 'ecg_hr_std', 'ecg_hr_min', 'ecg_hr_max', 'ecg_hr_p25', 'ecg_hr_p75', 'ecg_hr_skew', 'ecg_hr_kurtosis']
['ibi_mean', 'ibi_std', 'ibi_min', 'ibi_max', 'ibi_p25', 'ibi_p75', 'ibi_skew', 'ibi_kurtosis', 'rmssd_mean', 'rmssd_std', 'rmssd_min', 'rmssd_max', 'rmssd_p25', 'rmssd_p75', 'rmssd_skew', 'rmssd_kurtosis', 'sdnn_mean', 'sdnn_std', 'sdnn_min', 'sdnn_max', 'sdnn_p25', 'sdnn_p75', 'sdnn_skew', 'sdnn_kurtosis', 'ecg_ibi_mean', 'ecg_ibi_std', 'ecg_ibi_min', 'ecg_ibi_max', 'ecg_ibi_p25', 'ecg_ibi_p75', 'ecg_ibi_skew', 'ecg_ibi_kurtosis', 'ecg_rmssd_mean', 'ecg_rmssd_std', 'ecg_rmssd_min', 'ecg_rmssd_max', 

In [6]:
demographic_cols = ['age', 'gender', 'coffee_last_hour', 'sports', 'smoker', 'ill']

selected_features = [col for col in feature_cols if any(
    col.startswith(sig) for sig in ['acc_wrist', 'hr_', 'ibi_', 'rmssd_', 'sdnn_']
)] + [c for c in demographic_cols if c in feature_cols]

print(f"{len(selected_features)} features no total")

46 features no total


In [ ]:
X = X[selected_features]
logo = LeaveOneGroupOut()

In [8]:
results = []
for TOP_K in [None]:  # None = todas as features (baseline)

    all_true = []
    all_pred = []
    fold_f1s = []

    for train_idx, test_idx in logo.split(X, y, groups):

        X_train = X.iloc[train_idx];  X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx];  y_test = y.iloc[test_idx]

        scaler  = StandardScaler()
        imputer = SimpleImputer(strategy='mean')
        X_train_proc = imputer.fit_transform(scaler.fit_transform(X_train))
        X_test_proc  = imputer.transform(scaler.transform(X_test))

        if TOP_K is None:
            X_train_sel = X_train_proc
            X_test_sel  = X_test_proc
        else:
            ig_scores = mutual_info_classif(X_train_proc, y_train, random_state=42)
            top_k_idx = np.argsort(ig_scores)[::-1][:TOP_K]
            X_train_sel = X_train_proc[:, top_k_idx]
            X_test_sel  = X_test_proc[:, top_k_idx]

        X_train_sm, y_train_sm = SMOTE(random_state=42).fit_resample(
            X_train_sel, y_train
        )

        estimators = [
            ('xgb', XGBClassifier(
                n_estimators=200, max_depth=6, learning_rate=0.1,
                n_jobs=-1, random_state=42, eval_metric='mlogloss'
            )),
            ('rf', RandomForestClassifier(
                n_estimators=200, min_samples_split=10, min_samples_leaf=1,
                max_features='sqrt', max_depth=10,
                n_jobs=-1, random_state=42
            )),
            ('lgbm', LGBMClassifier(
                n_estimators=305, max_depth=4, learning_rate=0.195,
                num_leaves=37, min_child_samples=30,
                subsample=0.6537551310524808, colsample_bytree=0.8522263467378263,
                n_jobs=-1, random_state=42, verbose=-1
            )),
        ]

        ensemble = VotingClassifier(estimators=estimators, voting='soft')
        ensemble.fit(X_train_sm, y_train_sm)

        y_pred = ensemble.predict(X_test_sel)
        all_true.extend(y_test)
        all_pred.extend(y_pred)
        fold_f1s.append(f1_score(y_test, y_pred, average='macro', zero_division=0))

    results.append({
        'TOP_K'       : TOP_K if TOP_K is not None else 'All',
        'Macro_F1'    : f1_score(all_true, all_pred, average='macro', zero_division=0),
        'Accuracy'    : accuracy_score(all_true, all_pred),
        'Bal. Acc'    : balanced_accuracy_score(all_true, all_pred),
        'Precision'   : precision_score(all_true, all_pred, average='macro', zero_division=0),
        'Recall'      : recall_score(all_true, all_pred, average='macro', zero_division=0),
        'Std_F1'      : np.std(fold_f1s),
        'Mean_Fold_F1': np.mean(fold_f1s),
    })
    print(f"TOP_K={TOP_K} - Macro F1: {results[-1]['Macro_F1']:.4f} | "
          f"Acc: {results[-1]['Accuracy']:.4f} | Std: {results[-1]['Std_F1']:.4f}")

df_results = pd.DataFrame(results).set_index('TOP_K')
print("\n", df_results.to_string())

NameError: name 'logo' is not defined

In [ ]:
label = "todas (baseline)" if TOP_K is None else f"TOP_{TOP_K}"

print(f"\n{'='*55}")
print(f"  TOP_K = {label}")
print(f"{'='*55}")
print(classification_report(all_true, all_pred,
                            target_names=['baseline', 'stress'],
                            zero_division=0))

macro_f1  = np.mean(fold_f1s)
std_f1    = np.std(fold_f1s)
acc       = accuracy_score(all_true, all_pred)
bal_acc   = balanced_accuracy_score(all_true, all_pred)
precision = precision_score(all_true, all_pred, average='macro', zero_division=0)
recall    = recall_score(all_true, all_pred, average='macro', zero_division=0)

print(f"Accuracy:              {acc:.3f}")
print(f"Balanced Accuracy:     {bal_acc:.3f}")
print(f"Macro Precision:       {precision:.3f}")
print(f"Macro Recall:          {recall:.3f}")
print(f"Macro F1 médio:        {macro_f1:.3f} ± {std_f1:.3f}")

results.append({
    'TOP_K':      label,
    'Accuracy':   acc,
    'Bal. Acc':   bal_acc,
    'Precision':  precision,
    'Recall':     recall,
    'Macro F1':   macro_f1,
    'Std F1':     std_f1,
})

# matriz de confusão por TOP_K
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(all_true, all_pred),
            annot=True, fmt='d', cmap='Blues',
            xticklabels=['baseline', 'stress'],
            yticklabels=['baseline', 'stress', 'amusement'])
plt.title(f'Confusion Matrix — {label}')
plt.ylabel('Real'); plt.xlabel('Predito')
plt.tight_layout()
plt.show()

In [ ]:
results = [r for r in results if r['TOP_K'] in [10, 20, 30, 'All']]
print("\n\nRESUMO COMPARATIVO")
print("="*75)
df_results = pd.DataFrame(results).set_index('TOP_K')
print(df_results.to_string())

In [ ]:
feature_names = np.array(selected_features) 

fi_df = pd.DataFrame({
    'XGB' : ensemble.named_estimators_['xgb'].feature_importances_,
    'RF'  : ensemble.named_estimators_['rf'].feature_importances_,
    'LGBM': ensemble.named_estimators_['lgbm'].feature_importances_,
}, index=feature_names)

# Normaliza por modelo antes de fazer média
fi_df = fi_df / fi_df.sum()
mean_importance = fi_df.mean(axis=1).sort_values()

plt.figure(figsize=(20, 19))
plt.barh(mean_importance.index, mean_importance.values)
plt.title("Feature Importance Média — Ensemble Heterogéneo (todas as features)")
plt.tight_layout()
plt.show()

In [ ]:
quest_cols = ['age', 'gender', 'coffee_last_hour', 'sports', 'smoker', 'ill']
quest_indices = [list(selected_features).index(c) for c in quest_cols]

fi_mean = np.array([
    ensemble.named_estimators_['xgb'].feature_importances_,
    ensemble.named_estimators_['rf'].feature_importances_,
    ensemble.named_estimators_['lgbm'].feature_importances_,
])
fi_mean = (fi_mean / fi_mean.sum(axis=1, keepdims=True)).mean(axis=0)

quest_importances = fi_mean[quest_indices]
indices = np.argsort(quest_importances)

plt.figure(figsize=(8, 5))
plt.barh(np.array(quest_cols)[indices], quest_importances[indices])
plt.title("Feature Importance — Questionário (Média Ensemble)")
plt.tight_layout()
plt.show()

In [ ]:
def get_importance(model, name):
    if hasattr(model, 'feature_importances_'):
        return model.feature_importances_
    elif hasattr(model, 'coef_'):
        # LR com l1/l2: coef_ shape (1, n_features) ou (n_classes, n_features)
        return np.abs(model.coef_).mean(axis=0)
    else:
        return None  # SVM poly - sem importância direta

In [ ]:
available = {}
for name in ['xgb', 'rf', 'lgbm']:
    imp = get_importance(ensemble.named_estimators_[name], name)
    if imp is not None:
        imp = imp / imp.sum()
        available[name.upper()] = imp
        
for name, col in available.items():
    print(f"{name}: max={col.max():.4f}, std={col.std():.4f}, nonzero={np.sum(col > 0)}/{len(col)}")

## Ensemble with All Models

In [ ]:
results = []

for TOP_K in [None]:  # None = todas as features (baseline)
    all_true = []
    all_pred = []
    fold_f1s = []

    for train_idx, test_idx in logo.split(X, y, groups):
        X_train = X.iloc[train_idx];  X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx];  y_test = y.iloc[test_idx]

        scaler  = StandardScaler()
        imputer = SimpleImputer(strategy='mean')
        X_train_proc = imputer.fit_transform(scaler.fit_transform(X_train))
        X_test_proc  = imputer.transform(scaler.transform(X_test))

        if TOP_K is None:
            X_train_sel = X_train_proc
            X_test_sel  = X_test_proc
        else:
            ig_scores = mutual_info_classif(X_train_proc, y_train, random_state=42)
            top_k_idx = np.argsort(ig_scores)[::-1][:TOP_K]
            X_train_sel = X_train_proc[:, top_k_idx]
            X_test_sel  = X_test_proc[:, top_k_idx]

        X_train_sm, y_train_sm = SMOTE(random_state=42).fit_resample(
            X_train_sel, y_train
        )

        estimators = [
            ('xgb', XGBClassifier(
                n_estimators=200, max_depth=6, learning_rate=0.1,
                n_jobs=-1, random_state=42, eval_metric='mlogloss'
            )),
            ('rf', RandomForestClassifier(
                n_estimators=200, min_samples_split=10, min_samples_leaf=1,
                max_features='sqrt', max_depth=10,
                n_jobs=-1, random_state=42
            )),
            ('lgbm', LGBMClassifier(
                n_estimators=305, max_depth=4, learning_rate=0.195,
                num_leaves=37, min_child_samples=30,
                subsample=0.6537551310524808, colsample_bytree=0.8522263467378263,
                n_jobs=-1, random_state=42, verbose=-1
            )),
            ('lr', LogisticRegression(
                C=0.1, max_iter=3000, penalty='l1',
                random_state=42, solver='saga'
            )),
            ('svm', SVC(
                C=0.1, kernel='poly', gamma='auto', degree=3,
                class_weight='balanced', probability=True  
            )),
        ]

        ensemble = VotingClassifier(estimators=estimators, voting='soft')
        ensemble.fit(X_train_sm, y_train_sm)

        y_pred = ensemble.predict(X_test_sel)
        all_true.extend(y_test)
        all_pred.extend(y_pred)
        fold_f1s.append(f1_score(y_test, y_pred, average='macro', zero_division=0))

    results.append({
        'TOP_K'       : TOP_K if TOP_K is not None else 'All',
        'Macro_F1'    : f1_score(all_true, all_pred, average='macro', zero_division=0),
        'Accuracy'    : accuracy_score(all_true, all_pred),
        'Bal. Acc'    : balanced_accuracy_score(all_true, all_pred),
        'Precision'   : precision_score(all_true, all_pred, average='macro', zero_division=0),
        'Recall'      : recall_score(all_true, all_pred, average='macro', zero_division=0),
        'Std_F1'      : np.std(fold_f1s),
        'Mean_Fold_F1': np.mean(fold_f1s),
    })
    print(f"TOP_K={TOP_K} - Macro F1: {results[-1]['Macro_F1']:.4f} | "
          f"Acc: {results[-1]['Accuracy']:.4f} | Std: {results[-1]['Std_F1']:.4f}")

print("\n\nRESUMO COMPARATIVO")
print("="*75)
df_results = pd.DataFrame(results).set_index('TOP_K')
print(df_results.to_string())

In [ ]:
label = "todas (baseline)" if TOP_K is None else f"TOP_{TOP_K}"

print(f"\n{'='*55}")
print(f"  TOP_K = {label}")
print(f"{'='*55}")
print(classification_report(all_true, all_pred,
                            target_names=['baseline', 'stress'],
                            zero_division=0))

macro_f1  = np.mean(fold_f1s)
std_f1    = np.std(fold_f1s)
acc       = accuracy_score(all_true, all_pred)
bal_acc   = balanced_accuracy_score(all_true, all_pred)
precision = precision_score(all_true, all_pred, average='macro', zero_division=0)
recall    = recall_score(all_true, all_pred, average='macro', zero_division=0)

print(f"Accuracy:              {acc:.3f}")
print(f"Balanced Accuracy:     {bal_acc:.3f}")
print(f"Macro Precision:       {precision:.3f}")
print(f"Macro Recall:          {recall:.3f}")
print(f"Macro F1 médio:        {macro_f1:.3f} ± {std_f1:.3f}")

results.append({
    'TOP_K':      label,
    'Accuracy':   acc,
    'Bal. Acc':   bal_acc,
    'Precision':  precision,
    'Recall':     recall,
    'Macro F1':   macro_f1,
    'Std F1':     std_f1,
})

# matriz de confusão por TOP_K
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(all_true, all_pred),
            annot=True, fmt='d', cmap='Blues',
            xticklabels=['baseline', 'stress'],
            yticklabels=['baseline', 'stress', 'amusement'])
plt.title(f'Confusion Matrix — {label}')
plt.ylabel('Real'); plt.xlabel('Predito')
plt.tight_layout()
plt.show()

In [ ]:
label = "todas (baseline)" if TOP_K is None else f"TOP_{TOP_K}"

print(f"\n{'='*55}")
print(f"  TOP_K = {label}")
print(f"{'='*55}")
print(classification_report(all_true, all_pred,
                            target_names=['baseline', 'stress'],
                            zero_division=0))

macro_f1  = np.mean(fold_f1s)
std_f1    = np.std(fold_f1s)
acc       = accuracy_score(all_true, all_pred)
bal_acc   = balanced_accuracy_score(all_true, all_pred)
precision = precision_score(all_true, all_pred, average='macro', zero_division=0)
recall    = recall_score(all_true, all_pred, average='macro', zero_division=0)

print(f"Accuracy:              {acc:.3f}")
print(f"Balanced Accuracy:     {bal_acc:.3f}")
print(f"Macro Precision:       {precision:.3f}")
print(f"Macro Recall:          {recall:.3f}")
print(f"Macro F1 médio:        {macro_f1:.3f} ± {std_f1:.3f}")

results.append({
    'TOP_K':      label,
    'Accuracy':   acc,
    'Bal. Acc':   bal_acc,
    'Precision':  precision,
    'Recall':     recall,
    'Macro F1':   macro_f1,
    'Std F1':     std_f1,
})

# matriz de confusão por TOP_K
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(all_true, all_pred),
            annot=True, fmt='d', cmap='Blues',
            xticklabels=['baseline', 'stress'],
            yticklabels=['baseline', 'stress', 'amusement'])
plt.title(f'Confusion Matrix — {label}')
plt.ylabel('Real'); plt.xlabel('Predito')
plt.tight_layout()
plt.show()

In [ ]:
# modelos que têm importância diretas
available = {}
for name in ['xgb', 'rf', 'lgbm', 'lr']:
    imp = get_importance(ensemble.named_estimators_[name], name)
    if imp is not None:
        imp = imp / imp.sum()
        available[name.upper()] = imp

fi_df = pd.DataFrame(available, index=feature_names)
mean_importance = fi_df.mean(axis=1).sort_values()

plt.figure(figsize=(20, 19))
plt.barh(mean_importance.index, mean_importance.values)
plt.title("Feature Importance Média — Ensemble (XGB + RF + LGBM + LR)")
plt.tight_layout()
plt.show()

In [ ]:
fi_array = np.array([fi_df[m].values for m in fi_df.columns])  # já normalizado
fi_mean_quest = fi_array.mean(axis=0)

quest_cols = ['age', 'gender', 'coffee_last_hour', 'sports', 'smoker', 'ill']
quest_indices = [list(selected_features).index(c) for c in quest_cols]
quest_importances = fi_mean_quest[quest_indices]
indices = np.argsort(quest_importances)

plt.figure(figsize=(8, 5))
plt.barh(np.array(quest_cols)[indices], quest_importances[indices])
plt.title("Feature Importance — Questionário (Média XGB + RF + LGBM + LR)")
plt.tight_layout()
plt.show()

In [ ]:
for name, col in available.items():
    print(f"{name}: max={col.max():.4f}, std={col.std():.4f}, nonzero={np.sum(col > 0)}/{len(col)}")

In [ ]:
# ROC-AUC
from sklearn.metrics import roc_auc_score
import numpy as np
_auc = roc_auc_score(np.array(all_true), np.array(y_score))
print(f'ROC-AUC: {_auc:.3f}')

In [ ]:
# F1 @ Youden-optimal threshold
from sklearn.metrics import roc_curve, f1_score
import numpy as np
_yt = np.array(all_true); _ys = np.array(y_score)
_fpr, _tpr, _thr = roc_curve(_yt, _ys)
_t_opt = float(_thr[np.argmax(_tpr - _fpr)])
_f1opt = f1_score(_yt, (_ys >= _t_opt).astype(int), average='macro', zero_division=0)
print(f'F1@opt : {_f1opt:.3f}  (threshold={_t_opt:.3f})')